## Import Libraries

In [1]:
import numpy as np
import pandas as pd
import re
import ast
import nltk
from nltk.tokenize import sent_tokenize
from nltk.sentiment import SentimentIntensityAnalyzer
import warnings
warnings.filterwarnings("ignore")

In [2]:
## Download toolkit

In [3]:
nltk.download('vader_lexicon')
nltk.download('punkt_tab')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Asus\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Asus\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Load Dataset

In [4]:
zomato_data = pd.read_csv(r"C:\Users\Asus\Documents\NLP_based_resturant_recomendation_system\data\processed\restaurants_cleaned.csv")

## Dataset Columns & Info

In [5]:
zomato_data.columns

Index(['address', 'name', 'online_order', 'book_table', 'rate', 'votes',
       'phone', 'location', 'rest_type', 'dish_liked', 'cuisines',
       'approx_cost(for two people)', 'reviews_list', 'listed_in(type)'],
      dtype='object')

In [6]:
zomato_data.shape

(51717, 14)

In [7]:
zomato_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51717 entries, 0 to 51716
Data columns (total 14 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   address                      51717 non-null  object 
 1   name                         51717 non-null  object 
 2   online_order                 51717 non-null  int64  
 3   book_table                   51717 non-null  int64  
 4   rate                         51717 non-null  float64
 5   votes                        51717 non-null  int64  
 6   phone                        50509 non-null  object 
 7   location                     51717 non-null  object 
 8   rest_type                    51490 non-null  object 
 9   dish_liked                   51717 non-null  object 
 10  cuisines                     51717 non-null  object 
 11  approx_cost(for two people)  51717 non-null  float64
 12  reviews_list                 51717 non-null  object 
 13  listed_in(type) 

In [8]:
zomato_data.isnull().sum()

address                           0
name                              0
online_order                      0
book_table                        0
rate                              0
votes                             0
phone                          1208
location                          0
rest_type                       227
dish_liked                        0
cuisines                          0
approx_cost(for two people)       0
reviews_list                      0
listed_in(type)                   0
dtype: int64

In [9]:
 zomato_data.head()

,address,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,cuisines,approx_cost(for two people),reviews_list,listed_in(type)
0,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,1,1,4.1,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,"['Pasta', 'Lunch Buffet', 'Masala Papad', 'Pan...","['North Indian', 'Mughlai', 'Chinese']",800.0,"[('Rated 4.0', 'RATED\n A beautiful place to ...",Buffet
1,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,1,0,4.1,787,080 41714161,Banashankari,Casual Dining,"['Momos', 'Lunch Buffet', 'Chocolate Nirvana',...","['Chinese', 'North Indian', 'Thai']",800.0,"[('Rated 4.0', 'RATED\n Had been here for din...",Buffet
2,"1112, Next to KIMS Medical College, 17th Cross...",San Churro Cafe,1,0,3.8,918,+91 9663487993,Banashankari,"Cafe, Casual Dining","['Churros', 'Cannelloni', 'Minestrone Soup', '...","['Cafe', 'Mexican', 'Italian']",800.0,"[('Rated 3.0', ""RATED\n Ambience is not that ...",Buffet
3,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",Addhuri Udupi Bhojana,0,0,3.7,88,+91 9620009302,Banashankari,Quick Bites,['Masala Dosa'],"['South Indian', 'North Indian']",300.0,"[('Rated 4.0', ""RATED\n Great food and proper...",Buffet
4,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",Grand Village,0,0,3.8,166,+91 8026612447\r\n+91 9901210005,Basavanagudi,Casual Dining,"['Panipuri', 'Gol Gappe']","['North Indian', 'Rajasthani']",600.0,"[('Rated 4.0', 'RATED\n Very good restaurant ...",Buffet


## Step 1: Create Review Extraction Function

In [10]:
def extract_reviews_and_ratings(review_list):

    if pd.isna(review_list):
        return pd.Series([[], []])

    try:

        reviews = ast.literal_eval(review_list)

        ratings = []
        review_texts = []

        for review in reviews:

            if len(review) < 2:
                continue

            rating_text = review[0]
            review_text = review[1]

            # Extract numeric rating
            match = re.search(
                r"(\d+(?:\.\d+)?)",
                rating_text
            )

            if match:
                ratings.append(
                    float(match.group(1))
                )
            else:
                ratings.append(np.nan)

            review_texts.append(
                review_text
            )

        return pd.Series(
            [ratings, review_texts]
        )

    except:

        return pd.Series([[], []])


In [11]:
zomato_data[
    ["review_ratings", "review_texts"]
] = zomato_data["reviews_list"].apply(
    extract_reviews_and_ratings
)

## Step 2: Remove RATED

In [12]:
def remove_rated(text):

    text = re.sub(
        r"\bRATED\b",
        "",
        text,
        flags=re.IGNORECASE
    )

    return text

## Step 3: Remove HTML

In [13]:
def remove_html(text):

    return re.sub(r"<.*?>", " ", text)

## Step 4: Remove URLs

In [14]:
def remove_urls(text):

    return re.sub(
        r"http\S+|www\S+",
        "",
        text
    )

## Step 5: Remove Emojis

In [15]:
def remove_emoji(text):

    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "]+",
        flags=re.UNICODE
    )

    return emoji_pattern.sub("", text)

## Step 6: Remove Special Characters

In [16]:
def remove_special_chars(text):

    return re.sub(
        r"[^a-zA-Z0-9\s,.!?]",
        " ",
        text
    )

## Step 7: Normalize Whitespace

In [17]:
def normalize_spaces(text):

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()

## Step 8: Complete Cleaning Pipeline

In [18]:
def clean_review_text(text):

    text = str(text)

    text = remove_rated(text)

    text = remove_html(text)

    text = remove_urls(text)

    text = remove_emoji(text)

    text = remove_special_chars(text)

    text = text.lower()

    text = normalize_spaces(text)

    return text

In [19]:
zomato_data["review_texts"] = (
    zomato_data["review_texts"]
    .apply(
        lambda reviews:
        [
            clean_review_text(review)
            for review in reviews
        ]
    )
)

## Calculate review_count

In [84]:
zomato_data["review_count"] = (
    zomato_data["review_texts"]
    .apply(len)
)

## Sentiment Score (using VADER)

In [21]:
sia = SentimentIntensityAnalyzer()

## Create Review Sentiment Function

In [22]:
def get_sentiment_score(review):

    if not isinstance(review, str):
        return 0

    return sia.polarity_scores(review)["compound"]
    

## Calculate Review-Level Sentiment

In [23]:
zomato_data["review_sentiments"] = (
    zomato_data["review_texts"]
    .apply(
        lambda reviews:
        [
            get_sentiment_score(review)
            for review in reviews
        ]
    )
)

In [24]:
zomato_data[["review_texts", "review_sentiments"]].sample(20)

,review_texts,review_sentiments
11503,[],[]
25533,[the cafe is located on the main road so its e...,[0.9657]
34804,[the only sad part is i ordered one burger and...,"[0.4215, 0.97, 0.9834, 0.7506, -0.7269, 0.984,..."
33599,"[i loved the food here, it is good and customi...",[0.9788]
3358,[pocket friendly i have tasted almost all the ...,"[0.9179, -0.8611, 0.836, -0.6249, 0.0772, 0.82..."
12280,[],[]
45658,"[small, self service restaurant. doesn t offer...","[-0.8201, 0.8047, 0.91, 0.9062, 0.7096, -0.455..."
40427,[a typical military hotel located near ktm sho...,"[0.9821, 0.7783]"
29206,[the most underrated place in bangalore! i had...,"[0.9293, 0.7843, 0.658, 0.9834, 0.9872, 0.8331..."
20149,[ambience is good with the well chilled place ...,[0.891]


## Restaurant Sentiment Score

In [25]:
zomato_data["sentiment_score"] = (
    zomato_data["review_sentiments"]
    .apply(
        lambda scores:
        np.mean(scores)
        if len(scores) > 0
        else np.nan
    )
)

In [26]:
zomato_data["sentiment_score"] = (
    zomato_data["sentiment_score"] + 1
) / 2

In [27]:
zomato_data[["review_texts", "review_sentiments", "sentiment_score"]].sample(20)

,review_texts,review_sentiments,sentiment_score
32002,[fantastic food. good size portions. one star ...,"[0.8698, 0.9847, 0.9485, 0.9487, 0.989, 0.9099...",0.971950
22746,[],[],NaN
23383,[],[],NaN
34998,[had original glazed donuts and for some reaso...,"[0.6369, 0.7995]",0.859100
28577,[ordered schezwan noodles with mixed veg manch...,[-0.7405],0.129750
21128,[this is a not much known place in jayanagar. ...,"[0.9319, -0.958, -0.8611, 0.985, -0.2709]",0.482690
61,[1 food very nice 2 staff very good 3 ambience...,"[0.9575, 0.6249, 0.9796, 0.9714]",0.941675
3860,[love biryani? order from potful and you will ...,"[0.875, 0.9509, 0.7819, 0.8686, -0.5399, 0.96,...",0.870742
37031,[ordered butter chicken combo from this place ...,[-0.7553],0.122350
19896,[this restaurant is our breakfast and lunch pl...,"[0.895, 0.9449]",0.959975


## Calculate popularity_score

In [29]:
from sklearn.preprocessing import MinMaxScaler

zomato_data["popularity_score"] = (
    zomato_data["rate"]
    * np.log1p(zomato_data["votes"])
)

scaler = MinMaxScaler()

zomato_data["popularity_score"] = (
    scaler.fit_transform(
        zomato_data[["popularity_score"]]
    )
)

## Create Food Keyword Dictionary

In [30]:
food_keywords = {
    "food",
    "taste",
    "tasty",
    "delicious",
    "dish",
    "meal",
    "biryani",
    "pizza",
    "burger",
    "dessert",
    "flavor",
    "flavour",
    "fresh",
    "cuisine",
    "starter",
    "main course",
    "portion"
}

## Extract Food Related Sentences

In [31]:
def get_food_sentences(review, food_keywords):

    food_sentences = []

    sentences = sent_tokenize(review)

    for sentence in sentences:

        sentence_lower = sentence.lower()

        if any(
            keyword in sentence_lower
            for keyword in food_keywords
        ):
            food_sentences.append(sentence)

    return food_sentences

## Calculate Food Sentiment Score Per Review

In [32]:
def calculate_food_score(review):

    food_sentences = get_food_sentences(
        review,
        food_keywords
    )

    if len(food_sentences) == 0:
        return np.nan

    scores = []

    for sentence in food_sentences:

        score = sia.polarity_scores(
            sentence
        )["compound"]

        scores.append(score)

    return np.mean(scores)

In [33]:
zomato_data["food_scores_review"] = (
    zomato_data["review_texts"]
    .apply(
        lambda reviews:
        [
            calculate_food_score(review)
            for review in reviews
        ]
    )
)

## Calculate Restaurant Food Score

In [34]:
def aggregate_food_score(scores):

    scores = [
        score
        for score in scores
        if pd.notna(score)
    ]

    if len(scores) == 0:
        return np.nan

    return np.mean(scores)

In [35]:
zomato_data["food_score"] = (
    zomato_data["food_scores_review"]
    .apply(
        aggregate_food_score
    )
)

## Normalize to 0–1

VADER:

-1 → negative
+1 → positive

In [36]:
zomato_data["food_score"] = (
    zomato_data["food_score"] + 1
) / 2

## Food Mention Count

In [85]:
def count_food_mentions(review):

    review = review.lower()

    return sum(
        keyword in review
        for keyword in food_keywords
    )

In [86]:
zomato_data["food_mentions"] = (
    zomato_data["review_texts"]
    .apply(
        lambda reviews:
        sum(
            count_food_mentions(review)
            for review in reviews
        )
    )
)

## Create Ambiance Keyword Dictionary

In [ ]:
ambiance_keywords = {
    "ambience",
    "ambiance",
    "atmosphere",
    "decor",
    "interior",
    "interiors",
    "lighting",
    "music",
    "seating",
    "view",
    "environment",
    "vibe",
    "vibes",
    "crowd",
    "rooftop",
    "peaceful",
    "cozy",
    "comfortable",
    "romantic",
    "spacious"
}

## Extract Ambiance Sentences

In [ ]:
def get_ambiance_sentences(review):

    ambiance_sentences = []

    sentences = sent_tokenize(review)

    for sentence in sentences:

        sentence_lower = sentence.lower()

        if any(
            keyword in sentence_lower
            for keyword in ambiance_keywords
        ):
            ambiance_sentences.append(sentence)

    return ambiance_sentences

## Calculate Ambiance Score For One Review

In [ ]:
def calculate_ambiance_score(review):

    ambiance_sentences = get_ambiance_sentences(
        review
    )

    if len(ambiance_sentences) == 0:
        return np.nan

    scores = []

    for sentence in ambiance_sentences:

        score = sia.polarity_scores(
            sentence
        )["compound"]

        scores.append(score)

    return np.mean(scores)

## Calculate Review-Level Ambiance Scores

In [ ]:
zomato_data["ambiance_scores_review"] = (
    zomato_data["review_texts"]
    .apply(
        lambda reviews:
        [
            calculate_ambiance_score(review)
            for review in reviews
        ]
    )
)

## Aggregate To Restaurant Level

In [ ]:
def aggregate_ambiance_score(scores):

    scores = [
        score
        for score in scores
        if pd.notna(score)
    ]

    if len(scores) == 0:
        return np.nan

    return np.mean(scores)

In [ ]:
zomato_data["ambiance_score_raw"] = (
    zomato_data["ambiance_scores_review"]
    .apply(
        aggregate_ambiance_score
    )
)

## Normalize Between 0 and 1

In [45]:
zomato_data["ambiance_score"] = (
    zomato_data["ambiance_score_raw"] + 1
) / 2

## Ambiance Mention Count

In [46]:
def count_ambiance_mentions(review):

    review = review.lower()

    return sum(
        keyword in review
        for keyword in ambiance_keywords
    )

In [47]:
zomato_data["ambiance_mentions"] = (
    zomato_data["review_texts"]
    .apply(
        lambda reviews:
        sum(
            count_ambiance_mentions(review)
            for review in reviews
        )
    )
)

## Create Authenticity Keywords

In [48]:
authenticity_keywords = {
    "authentic",
    "traditional",
    "original",
    "genuine",
    "heritage",
    "classic",
    "homestyle",
    "home style",
    "homemade",
    "native",
    "local cuisine",
    "real",
    "typical",
    "true taste",
    "signature",
    "andhra style",
    "south indian style",
    "north indian style"
}

## Extract Authenticity Sentences

In [49]:
def get_authenticity_sentences(review):

    authenticity_sentences = []

    sentences = sent_tokenize(review)

    for sentence in sentences:

        sentence_lower = sentence.lower()

        if any(
            keyword in sentence_lower
            for keyword in authenticity_keywords
        ):
            authenticity_sentences.append(
                sentence
            )

    return authenticity_sentences

## Calculate Authenticity Score Per Review

In [50]:
def calculate_authenticity_score(review):

    authenticity_sentences = (
        get_authenticity_sentences(review)
    )

    if len(authenticity_sentences) == 0:
        return np.nan

    scores = []

    for sentence in authenticity_sentences:

        score = sia.polarity_scores(
            sentence
        )["compound"]

        scores.append(score)

    return np.mean(scores)

In [51]:
zomato_data["authenticity_scores_review"] = (
    zomato_data["review_texts"]
    .apply(
        lambda reviews:
        [
            calculate_authenticity_score(review)
            for review in reviews
        ]
    )
)

## Aggregate Authenticity Score to Restaurant Level

In [52]:
def aggregate_authenticity_score(scores):

    scores = [
        score
        for score in scores
        if pd.notna(score)
    ]

    if len(scores) == 0:
        return np.nan

    return np.mean(scores)

In [53]:
zomato_data["authenticity_score_raw"] = (
    zomato_data["authenticity_scores_review"]
    .apply(
        aggregate_authenticity_score
    )
)

## Normalize Between 0 and 1

In [54]:
zomato_data["authenticity_score"] = (
    zomato_data["authenticity_score_raw"] + 1
) / 2

## Authenticity Mention Count

In [56]:
def count_authenticity_mentions(review):

    review = review.lower()

    return sum(
        keyword in review
        for keyword in authenticity_keywords
    )

In [57]:
zomato_data["authenticity_mentions"] = (
    zomato_data["review_texts"]
    .apply(
        lambda reviews:
        sum(
            count_authenticity_mentions(review)
            for review in reviews
        )
    )
)

## Define Service Keywords

In [58]:
service_keywords = {
    "service",
    "staff",
    "waiter",
    "waiters",
    "waitress",
    "manager",
    "hospitality",
    "courteous",
    "friendly",
    "attentive",
    "helpful",
    "professional",
    "polite",
    "behavior",
    "behaviour",
    "served",
    "serving",
    "server",
    "customer service"
}

## Extract Service-Related Sentences

In [59]:
def get_service_sentences(review):

    service_sentences = []

    sentences = sent_tokenize(review)

    for sentence in sentences:

        sentence_lower = sentence.lower()

        if any(
            keyword in sentence_lower
            for keyword in service_keywords
        ):
            service_sentences.append(sentence)

    return service_sentences

## Calculate Service Score Per Review

In [60]:
def calculate_service_score(review):

    service_sentences = get_service_sentences(
        review
    )

    if len(service_sentences) == 0:
        return np.nan

    scores = []

    for sentence in service_sentences:

        score = sia.polarity_scores(
            sentence
        )["compound"]

        scores.append(score)

    return np.mean(scores)

In [61]:
zomato_data["service_scores_review"] = (
    zomato_data["review_texts"]
    .apply(
        lambda reviews:
        [
            calculate_service_score(review)
            for review in reviews
        ]
    )
)

## Aggregate To Restaurant Level

In [62]:
def aggregate_service_score(scores):

    scores = [
        score
        for score in scores
        if pd.notna(score)
    ]

    if len(scores) == 0:
        return np.nan

    return np.mean(scores)

In [63]:
zomato_data["service_score_raw"] = (
    zomato_data["service_scores_review"]
    .apply(
        aggregate_service_score
    )
)

## Normalize Between 0 and 1

In [64]:
zomato_data["service_score"] = (
    zomato_data["service_score_raw"] + 1
) / 2

## Service Mention Count

In [65]:
def count_service_mentions(review):

    review = review.lower()

    return sum(
        keyword in review
        for keyword in service_keywords
    )

In [66]:
zomato_data["service_mentions"] = (
    zomato_data["review_texts"]
    .apply(
        lambda reviews:
        sum(
            count_service_mentions(review)
            for review in reviews
        )
    )
)

## Create Value-for-Money Keywords

In [68]:
value_keywords = {
    "worth",
    "worth it",
    "worth every penny",
    "affordable",
    "reasonable",
    "budget",
    "economical",
    "cheap",
    "value for money",
    "pocket friendly",
    "good value",
    "expensive",
    "overpriced",
    "costly",
    "pricey",
    "not worth",
    "waste of money"
}

## Extract Value-Related Sentences

In [69]:
def get_value_sentences(review):

    value_sentences = []

    sentences = sent_tokenize(review)

    for sentence in sentences:

        sentence_lower = sentence.lower()

        if any(
            keyword in sentence_lower
            for keyword in value_keywords
        ):
            value_sentences.append(sentence)

    return value_sentences

## Calculate Value Score For One Review

In [70]:
def calculate_value_score(review):

    value_sentences = get_value_sentences(
        review
    )

    if len(value_sentences) == 0:
        return np.nan

    scores = []

    for sentence in value_sentences:

        score = sia.polarity_scores(
            sentence
        )["compound"]

        scores.append(score)

    return np.mean(scores)

In [71]:
zomato_data["value_scores_review"] = (
    zomato_data["review_texts"]
    .apply(
        lambda reviews:
        [
            calculate_value_score(review)
            for review in reviews
        ]
    )
)

## Aggregate To Restaurant Level

In [72]:
def aggregate_value_score(scores):

    scores = [
        score
        for score in scores
        if pd.notna(score)
    ]

    if len(scores) == 0:
        return np.nan

    return np.mean(scores)

In [73]:
zomato_data["value_for_money_score_raw"] = (
    zomato_data["value_scores_review"]
    .apply(
        aggregate_value_score
    )
)

## Normalize Between 0 and 1

In [74]:
zomato_data["value_for_money_score"] = (
    zomato_data["value_for_money_score_raw"] + 1
) / 2

## Count Value Mentions

In [75]:
def count_value_mentions(review):

    review = review.lower()

    return sum(
        keyword in review
        for keyword in value_keywords
    )

In [76]:
zomato_data["value_mentions"] = (
    zomato_data["review_texts"]
    .apply(
        lambda reviews:
        sum(
            count_value_mentions(review)
            for review in reviews
        )
    )
)

In [78]:
zomato_data.to_csv(
    r"C:\Users\Asus\Documents\NLP_based_resturant_recomendation_system\data\processed\rest_data_with_all_cols.csv",
    index=False
)

## Drop columns not required
reviews_list, value_for_money_score_raw,value_scores_review, service_score_raw, service_scores_review, authenticity_score_raw, authenticity_score_review, ambiance_score_raw, ambiance_scores_review, food_scores_review, review_sentiments

In [79]:
columns_to_drop = [
    "reviews_list",
    "value_for_money_score_raw",
    "value_scores_review",
    "service_score_raw",
    "service_scores_review",
    "authenticity_score_raw",
    "authenticity_scores_review",
    "ambiance_score_raw",
    "ambiance_scores_review",
    "food_scores_review",
    "review_sentiments"
]

zomato_data.drop(
    columns=columns_to_drop,
    inplace=True,
    errors="ignore"
)

## Handle Missing Values

In [87]:
aspect_cols = [
    "sentiment_score",
    "food_score",
    "ambiance_score",
    "service_score",
    "authenticity_score",
    "value_for_money_score"
]

zomato_data[aspect_cols] = (
    zomato_data[aspect_cols]
    .fillna(0.5)
)

In [89]:
zomato_data.to_csv(
    r"C:\Users\Asus\Documents\NLP_based_resturant_recomendation_system\data\processed\restaurants_enriched.csv",
    index=False
)

In [90]:
zomato_data.head()

,address,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,...,food_score,food_mentions,ambiance_score,ambiance_mentions,authenticity_score,authenticity_mentions,service_score,service_mentions,value_for_money_score,value_mentions
0,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,1,1,4.1,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,"['Pasta', 'Lunch Buffet', 'Masala Papad', 'Pan...",...,0.723485,20,0.813145,14,0.907675,2,0.776086,12,0.806175,2
1,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,1,0,4.1,787,080 41714161,Banashankari,Casual Dining,"['Momos', 'Lunch Buffet', 'Chocolate Nirvana',...",...,0.706968,27,0.768720,13,0.747880,5,0.797105,15,0.639608,6
2,"1112, Next to KIMS Medical College, 17th Cross...",San Churro Cafe,1,0,3.8,918,+91 9663487993,Banashankari,"Cafe, Casual Dining","['Churros', 'Cannelloni', 'Minestrone Soup', '...",...,0.590025,33,0.573857,16,0.569633,3,0.600343,18,0.507917,3
3,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",Addhuri Udupi Bhojana,0,0,3.7,88,+91 9620009302,Banashankari,Quick Bites,['Masala Dosa'],...,0.700841,72,0.593326,19,0.816886,13,0.717199,55,0.646975,11
4,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",Grand Village,0,0,3.8,166,+91 8026612447\r\n+91 9901210005,Basavanagudi,Casual Dining,"['Panipuri', 'Gol Gappe']",...,0.674138,3,0.773325,1,0.500000,0,0.774050,3,0.746950,1


In [91]:
zomato_data.isnull().sum()

address                           0
name                              0
online_order                      0
book_table                        0
rate                              0
votes                             0
phone                          1208
location                          0
rest_type                       227
dish_liked                        0
cuisines                          0
approx_cost(for two people)       0
listed_in(type)                   0
review_ratings                    0
review_texts                      0
sentiment_score                   0
review_count                      0
popularity_score                  0
food_score                        0
food_mentions                     0
ambiance_score                    0
ambiance_mentions                 0
authenticity_score                0
authenticity_mentions             0
service_score                     0
service_mentions                  0
value_for_money_score             0
value_mentions              

## Validate Scores

Before building recommendations, verify the scores make sense.

In [92]:
zomato_data[
    ["name", "food_score"]
].sort_values(
    "food_score",
    ascending=False
).head(20)

,name,food_score
51151,Terracotta Whitefield,0.99715
31546,Kshera Sagar,0.99685
36180,Kshera Sagar,0.99685
27885,Kshera Sagar,0.99685
37415,Kshera Sagar,0.99685
30588,Kshera Sagar,0.99685
34348,Kshera Sagar,0.99685
28774,Kshera Sagar,0.99685
33450,Kshera Sagar,0.99685
19376,Mighty Small,0.99375


In [93]:
zomato_data[
    ["name", "ambiance_score"]
].sort_values(
    "ambiance_score",
    ascending=False
).head(20)

,name,ambiance_score
51151,Terracotta Whitefield,0.99715
39330,Matsuri - The Chancery Hotel,0.99475
43948,Matsuri - The Chancery Hotel,0.99475
43376,Matsuri - The Chancery Hotel,0.99475
38572,Matsuri - The Chancery Hotel,0.99475
48277,Matsuri - The Chancery Hotel,0.99475
21248,Hotel Paradise,0.99410
11331,Hotel Paradise,0.99410
46661,Buttercup Bakes,0.99390
37368,Hotel Paradise,0.99390


In [94]:
zomato_data[
    ["name", "authenticity_score"]
].sort_values(
    "authenticity_score",
    ascending=False
).head(20)

,name,authenticity_score
35278,Navami,0.99565
48277,Matsuri - The Chancery Hotel,0.99475
43376,Matsuri - The Chancery Hotel,0.99475
43948,Matsuri - The Chancery Hotel,0.99475
44997,Taiki,0.99395
13269,Mighty Small,0.99375
43833,Mighty Small,0.99375
12784,Mighty Small,0.99375
38829,Mighty Small,0.99375
48820,Mighty Small,0.99375
